# Exercise 10

## Group ID: 22
*   Taha El Amine Kassabi
*   Mohamed Hazem Badawi
*   Carolin Goj

## Exercise day: Tuesday

### Description

In this exercise, we will build a Multi-Layer Perceptron (MLP) to predict the next token. The task is divided into the following steps:

1. Define an MLP capable of predicting the next token. (4 points)

2. Implement a training loop for the MLP. (1 point)

This exercise provides an initial introduction to next-token prediction, which will be further explored in subsequent exercises.

Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from tqdm.auto import tqdm, trange

from torch.utils.data import Dataset, DataLoader

Let's download and load data that we will work on

In [2]:
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt -O ./data/names.txt

--2024-12-19 11:27:32--  https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘./data/names.txt’

./data/names.txt    100%[===================>] 222.80K  1.09MB/s    in 0.2s    

2024-12-19 11:27:33 (1.09 MB/s) - ‘./data/names.txt’ saved [228145/228145]



In [3]:
with open("./data/names.txt", "r", encoding="utf-8") as f:
    text = f.read()
names = text.split("\n")

print("# of names: ", len(names))
print(names[:10])

# of names:  32033
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


We create a text encoding by converting the text into a sequence of integers. This step is essential for preparing the text data to be processed by machine learning models, as they work with numerical data rather than raw text.


In [4]:
chars = sorted(set(text))
empty_char = '-'
chars.append(empty_char)
eol_char = '<'
chars.append(eol_char)

vocab_size = len(chars)

print(f'VOCAB: {"".join(chars)}')
print(f"\nVOCAB SIZE: {vocab_size}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s] if len(s) > 1 else stoi[s]
decode = lambda l: "".join([itos[i] for i in l]) if isinstance(l, list) else itos[l]

print(f"Encoding of hi there: {encode('emma')}")
print(f"Decoding the encoded message: {decode(encode('emma'))}")

VOCAB: 
abcdefghijklmnopqrstuvwxyz-<

VOCAB SIZE: 29
Encoding of hi there: [5, 13, 13, 1]
Decoding the encoded message: emma


We split the data into training and test sets.

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(names))
train_names = names[:n]
val_names = names[n:]
print((len(train_names), len(val_names)))

(28829, 3204)


Let's create a DataLoader for this data

In [6]:
blocksize = 8


class CustomTextDataset(Dataset):
    def __init__(self, data, blocksize):
        self.data = data
        self.blocksize = blocksize

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        name = self.data[idx]
        x_full = empty_char * self.blocksize + name + eol_char
        # Sample a random chunk of blocksize characters
        start_idx = torch.randint(0, len(x_full) - self.blocksize, (1,)).item()
        end_idx = start_idx + self.blocksize
        chunk = x_full[start_idx:end_idx]
        next_char = x_full[end_idx]
        # Encode the chunk and next_char
        return chunk, next_char


class CustomEncodedTextDataset(Dataset):
    def __init__(self, data, blocksize):
        self.data = CustomTextDataset(data, blocksize)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        chunk, next_char = self.data[idx]
        chunk_encoded = encode(chunk)
        next_char_encoded = encode(next_char)
        return torch.tensor(chunk_encoded, dtype=torch.long), torch.tensor(next_char_encoded, dtype=torch.long)


dataloader = DataLoader(CustomTextDataset(train_names, blocksize), batch_size=4)

train_dataset = CustomEncodedTextDataset(train_names, blocksize)
val_dataset = CustomEncodedTextDataset(val_names, blocksize)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [7]:
# sample from dataloader

for i, (x, y) in enumerate(dataloader):
    print(x)
    print(y)
    if i > 5:
        break

('------em', '------ol', '--------', '-----isa')
('m', 'i', 'a', 'b')
('---sophi', '--charlo', '-------m', '--------')
('a', 't', 'i', 'a')
('------ha', '----evel', '--------', '---emily')
('r', 'y', 'a', '<')
('--------', '----mila', '----ella', '------av')
('e', '<', '<', 'e')
('-------s', '-----cam', '--------', '--scarle')
('o', 'i', 'a', 't')
('victoria', '---madis', '----luna', '-----gra')
('<', 'o', '<', 'c')
('---chloe', '-----pen', '----layl', '----rile')
('<', 'e', 'a', 'y')


Define the MLP model for next-token prediction. This class should consist of the following inputs:

1. vocab_size - Represents the size of the vocabulary, i.e., the total number of unique tokens in the dataset.

2. block_size - The length of the input sequence considered by the model.

3. embedding_dim - The dimensionality of the embedding vectors. Each token in the input is mapped to a vector of this size.

4. hidden_dim - The number of neurons in the hidden layer of the MLP.

In [8]:
class MLPNextTokenPredictor(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(block_size * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        x = self.embedding(x)
        return self.layers(x)

This cell defines and implements the training loop for a next-token prediction model. 

In [9]:
model = MLPNextTokenPredictor(vocab_size, blocksize, 16, 32)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Here implement the training loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 5


@torch.inference_mode()
def validate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0
    val = tqdm(dataloader, desc='Validate', unit='batch')
    for x, y in val:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.size(0)
        val.set_postfix(acc=correct / total)


epochs = trange(num_epochs, desc='Epochs')
for epoch in epochs:
    model.train()
    running_loss = 0
    train = tqdm(train_loader, desc='Train', unit='batch')
    for x, y in train:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        batch_loss = criterion(pred, y)
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()
        running_loss += batch_loss.item()
        train.set_postfix(loss=batch_loss.item())

    validate(model, train_loader, device)
    validate(model, val_loader, device)

    average_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1} - Loss: {average_loss:.4f}")

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Train:   0%|          | 0/451 [00:00<?, ?batch/s]

Validate:   0%|          | 0/451 [00:00<?, ?batch/s]

Validate:   0%|          | 0/51 [00:00<?, ?batch/s]

Epoch 1 - Loss: 2.5850


...

Epoch 4 - Loss: 2.2555


Train:   0%|          | 0/451 [00:00<?, ?batch/s]

Validate:   0%|          | 0/451 [00:00<?, ?batch/s]

Validate:   0%|          | 0/51 [00:00<?, ?batch/s]

Epoch 5 - Loss: 2.2564


This function, generate, uses the trained model to generate a sequence of up to 100 tokens. Starting with an empty string of fixed length (blocksize), it iteratively predicts the next token, appends it to the sequence, and stops generation if an end-of-line character (eol_char) is encountered.

In [10]:
def generate(model):
    model.eval()
    s = empty_char * blocksize
    x = torch.tensor(encode(s), dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(100):
        logits = model(x[:, -blocksize:])
        next_token = torch.multinomial(F.softmax(logits, dim=-1), 1)
        s += itos[next_token.item()]
        x = torch.cat([x, next_token], dim=1)
        if s[-1] == eol_char:
            break
    s = s.replace(empty_char, '')
    s = s.replace(eol_char, '')
    return s

In [11]:
for _ in range(10):
    name = generate(model)
    print(name)

amyaea
myrion
ibiae
cavy
kareainaneva
kasera
deya
jolumen
jaemry
chasuel
